<a href="https://colab.research.google.com/github/Dennischegemwangi/code-base/blob/main/Building_a_transformer_from_scratch(encoder).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertviz transformers

In [ ]:
#visualizing attention
from transformers import AutoTokenizer
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model = BertModel.from_pretrained(model_ckpt)

text = "A car needs gas to run"
show(model,"bert",tokenizer,text,display_mode = "light",layer = 0,head = 8)

In [ ]:
#tokenization
input = tokenizer (text, return_tensors = "pt",add_special_tokens = False)

input.input_ids

tensor([[1037, 2482, 3791, 3806, 2000, 2448]])

In [ ]:
#creating embeddings

from torch import nn
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size,config.hidden_size)

#token_emb

#input embeddings
input_emb = token_emb(input.input_ids)
input_emb.size()

torch.Size([1, 6, 768])

In [ ]:
#creating query,key and value
import  torch
from math import sqrt

query = key = value = input_emb
dim_k = key.size(-1)
att_scores = torch.bmm(query,key.transpose(1,2))/sqrt(dim_k)

In [ ]:
#calculating weights
import torch.nn.functional as F
weights = F.softmax(att_scores,dim = -1)
weights.sum(dim = -1)

att_outputs = torch.bmm(weights, value)

In [ ]:
#scaled dot product attention

def SDPA (query,key,value):
    dim_k = key.size(-1)
    att_scores = torch.bmm(query,key.transpose(1,2))/sqrt(dim_k)
    weights = F.softmax(att_scores,dim = -1)
    return torch.bmm(weights, value)

In [ ]:
#multi-head attention implementation

#1 single attention head implementation
class AttentionHead(nn.Module):
      def __init__(self,embed_dim,head_dim):
          super().__init__()
          self.q = nn.Linear(embed_dim,head_dim)
          self.k = nn.Linear(embed_dim,head_dim)
          self.v = nn.Linear(embed_dim,head_dim)
      def forward(self, hidden_state):
          att_outputs = SDPA (self.q(hidden_state),self.k(hidden_state),self.v(hidden_state))

          return att_outputs

In [ ]:
#2 multiple attention heads

class multi_head_att(nn.Module):
      def __init__(self,config):

         super().__init__()
         embed_dim = config.hidden_size
         num_heads = config.num_attention_heads
         head_dim = embed_dim//num_heads

         self.heads = nn.ModuleList([AttentionHead(embed_dim,head_dim) for _ in range(num_heads)])
         self.output_linear = nn.Linear(embed_dim,embed_dim)
      def forward (self, hidden_state):
          x = torch.cat([h(hidden_state) for h in self.heads],dim = -1)
          x = self.output_linear(x)


          return x

In [ ]:
multihead_att = multi_head_att(config)
attn_outputs = multihead_att(input_emb)
attn_outputs.size()

torch.Size([1, 6, 768])

In [ ]:
#visualizing the multi head attention

from bertviz import head_view

from transformers import AutoModel


model = AutoModel.from_pretrained(model_ckpt, output_attentions = True)

sentence1 = "A car needs gas to run"

sentence2 = "Gas from burning plastic is toxic"

visualize_inputs = tokenizer(sentence1, sentence2, return_tensors = "pt")

attention = model(**visualize_inputs).attentions

sentence2_start = (visualize_inputs.token_type_ids == 0).sum(dim = 1)

tokens = tokenizer.convert_ids_to_tokens(visualize_inputs.input_ids[0])

head_view(attention, tokens, sentence2_start,heads = [8])

In [ ]:
#feed foward layer(for capacity and memorization)
class FeedForward (nn.Module):
      def __init__(self,config):
          super().__init__()
          self.linear_1 = nn.Linear(config.hidden_size,config.intermediate_size)
          self.linear_2 = nn.Linear(config.intermediate_size,config.hidden_size)
          self.gelu = nn.GELU()
          self.dropout = nn.Dropout(config.hidden_dropout_prob)

      def forward (self,x):
          x = self.linear_1(x)
          x = self.gelu(x)
          x = self.linear_2(x)
          x = self.dropout(x)

          return x

In [ ]:
feedforward = FeedForward (config)
ff_outputs = feedforward (att_outputs)

ff_outputs.size()

torch.Size([1, 6, 768])

In [ ]:
#adding layer normalization

class TransformerEncoderLayer(nn.Module):
      def __init__(self,config):
          super().__init__()
          self.layer1_norm = nn.LayerNorm(config.hidden_size)
          self.layer2_norm = nn.LayerNorm(config.hidden_size)
          self.attention = multi_head_att(config)
          self.feed_forward = FeedForward (config)

      def forward (self,x):
          hidden_state = self.layer1_norm(x)
          #apply skip connection
          x = x + self.layer2_norm(hidden_state)
          #feed forward with skip connection
          x = x + self.feed_forward(self.layer2_norm(x))
          return x

In [ ]:
encoder_layer = TransformerEncoderLayer (config)
input_emb.shape,encoder_layer(input_emb).size()

(torch.Size([1, 6, 768]), torch.Size([1, 6, 768]))

In [ ]:
#positional embeddings

class Embeddings (nn.Module):
      def __init__(self, config):
          super().__init__()
          self.token_emb = nn.Embedding(config.vocab_size,config.hidden_size)
          self.pos_emb = nn.Embedding(config.max_position_embeddings,config.hidden_size)
          self.layer_norm = nn.LayerNorm(config.hidden_size,eps = 1e-12)

          self.dropout = nn.Dropout()

      def forward (self,input_ids):
          #creating position ids for input sequence
          seq_len = input_ids.size(1)
          pos_ids = torch.arange(seq_len,dtype = torch.long).unsqueeze(0)
          #creating token and position embeddings

          token_emb = self.token_emb(input_ids)
          pos_emb = self.pos_emb(pos_ids)
          #combining token and position embeddings

          embeddings = token_emb + pos_emb
          embeddings = self.layer_norm(embeddings)
          embeddings = self.dropout(embeddings)

          return embeddings

In [ ]:
embedding_layer = Embeddings (config)
embedding_layer(input.input_ids).size()

torch.Size([1, 6, 768])

In [ ]:
#full transformer encoder

class TransformerEncoder(nn.Module):
      def __init__(self,config):
          super().__init__()
          self.embeddings = Embeddings (config)
          self.layers = nn.ModuleList([TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)])


      def forward (self,x):
          x = self.embeddings(x)
          for layer in self.layers:
              x = layer(x)
          return x

In [ ]:
encoder = TransformerEncoder (config)
encoder(input.input_ids).size()

torch.Size([1, 6, 768])

In [ ]:
#encoder for sequence classification
#adding classification head to a transformer

class TransformerForSeqClassification(nn.Module):
      def __init__(self,config):
          super().__init__()
          self.encoder = TransformerEncoder(config)
          self.dropout = nn.Dropout(config.hidden_dropout_prob)
          self.classifier = nn.Linear(config.hidden_size,config.num_labels)

      def forward (self,x):
          #select hidden state of the [cls] token
          x = self.encoder(x)[ :,0,:]
          x = self.dropout(x)
          x = self.classifier(x)

          return x

In [ ]:
config.num_labels = 3
encoder_classifier = TransformerForSeqClassification (config)
encoder_classifier(input.input_ids).size()

torch.Size([1, 3])